## Figure 15f — low-O3 Z300 and EP-flux composites

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: canonical figure15/merra2_bootstrap5000.nc and
figure15/waccm_bootstrap5000.nc. Paths are resolved through
PAPER1_DERIVED_ROOT with the repository-local diagnostics fallback.
Before plotting, the block validates product version, source label,
master-ranking provenance, master and field-available counts, inherited
low-25% flags, 5,000 trajectory resamples, ddof=0, and the EP-flux
method. The WACCM product fixes N=230/low=57 in the ranking master but
plots only the transparently recorded complete-field subset; it must identify
do_ubar=True, w=None, and natural-calendar monthly N2. Hatching marks
locations where the actual low-25% composite differs from the mean of
random composites by at least two bootstrap standard deviations.
Panels (b) and (d) omit grid lines and retain only a black 100-hPa
horizontal reference line.
MERRA-2 is above WACCM.

Outputs: figure15f_ubar_N2correction_bootstrap.png and PDF.

**Typography.** The plotting cell applies the shared publication-scale Paper 1 style immediately before saving: enlarged titles, axis labels, ticks, legends, and colour-bar text at the final manuscript canvas size.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    import sys as _sys
    style_directory = str(REPOSITORY_ROOT / "figures")
    if style_directory not in _sys.path:
        _sys.path.insert(0, style_directory)
    from paper_style import apply_paper_style
    apply_paper_style(figure, stem)
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
import matplotlib.path as mpath
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

REQUIRED = (
    "z300_low25_anomaly", "z300_stationary_climatology",
    "z300_bootstrap_significant", "epflux_low25_std_anomaly",
    "epflux_bootstrap_significant", "is_low25", "o3_minimum_du",
    "canonical_rank", "event_id",
)


def validated_figure15(relative, expected_source):
    product = load_dataset(relative, REQUIRED)
    if int(product.attrs.get("bootstrap_replicates", -1)) != 5000:
        raise ValueError(f"{relative}: expected 5000 bootstrap replicates")
    source_label = "".join(
        character for character in
        str(product.attrs.get("source_label", "")).lower()
        if character.isalnum()
    )
    expected_label = "".join(
        character for character in expected_source.lower()
        if character.isalnum()
    )
    if source_label != expected_label:
        raise ValueError(f"{relative}: unexpected source_label")
    for name in (
        "method", "master_ranking_path", "low25_definition",
        "low25_threshold_du", "bootstrap_method",
        "bootstrap_seed_z300", "bootstrap_seed_epflux",
        "epflux_method", "source_manifest", "master_sample_size",
        "master_low_count", "available_event_count",
        "available_low_count", "ep_standardization_ddof",
        "field_scope_segments", "scope_master_count",
        "scope_master_low_count", "excluded_master_event_ids",
        "natural_month_n2", "do_ubar", "w_argument", "wave",
        "z300_completeness",
    ):
        if str(product.attrs.get(name, "")).strip() == "":
            raise ValueError(f"{relative}: missing metadata {name}")
    expected_ranking = (
        "merra2_rankings.csv" if expected_label == "merra2"
        else "waccm_master_rankings.csv"
    )
    ranking_name = str(
        product.attrs["master_ranking_path"]
    ).replace("\\", "/").rsplit("/", 1)[-1]
    if ranking_name != expected_ranking:
        raise ValueError(
            f"{relative}: master_ranking_path ends in "
            f"{ranking_name!r}, expected {expected_ranking!r}"
        )
    ranking_path = canonical_path(f"ozone/{expected_ranking}")
    ranking = pd.read_csv(ranking_path)
    require_columns(
        ranking, ranking_path,
        (
            "event_id", "minimum_du", "rank", "is_low25",
            "low25_threshold_du", "sample_size", "low25_count",
            "source_segment",
        ),
    )
    if ranking["event_id"].astype(str).duplicated().any():
        raise ValueError(f"{ranking_path}: duplicate event_id")
    indexed = ranking.assign(
        event_id=ranking["event_id"].astype(str),
        is_low25=parse_boolean(ranking["is_low25"]),
    ).set_index("event_id")
    if expected_label == "waccm":
        scope = indexed[
            indexed["source_segment"].astype(str) == "LONGRUN"
        ]
        if len(scope) != 207:
            raise ValueError(f"{ranking_path}: expected 207 LONGRUN events")
    else:
        scope = indexed
    expected_excluded = sorted(set(indexed.index) - set(scope.index))
    recorded_excluded = sorted(filter(
        None,
        str(product.attrs.get("excluded_master_event_ids", "")).split(","),
    ))
    if recorded_excluded != expected_excluded:
        raise ValueError(
            f"{relative}: excluded master IDs differ from field scope"
        )
    expected_scope_segments = sorted(
        set(scope["source_segment"].astype(str))
    )
    recorded_scope_segments = sorted(filter(
        None,
        str(product.attrs["field_scope_segments"]).split(","),
    ))
    if recorded_scope_segments != expected_scope_segments:
        raise ValueError(f"{relative}: field-scope segments are wrong")
    if (
        int(product.attrs["scope_master_count"]) != len(scope)
        or int(product.attrs["scope_master_low_count"])
        != int(scope["is_low25"].sum())
    ):
        raise ValueError(f"{relative}: field-scope counts are wrong")
    product_ids = [
        text_value(value) for value in product["event_id"].values
    ]
    if len(product_ids) != len(set(product_ids)):
        raise ValueError(f"{relative}: duplicate available event_id")
    missing_ids = sorted(set(product_ids) - set(indexed.index))
    if missing_ids:
        raise ValueError(
            f"{relative}: event IDs absent from ranking {missing_ids[:5]}"
        )
    out_of_scope_ids = sorted(set(product_ids) - set(scope.index))
    if out_of_scope_ids:
        raise ValueError(
            f"{relative}: field product contains out-of-scope events "
            f"{out_of_scope_ids[:5]}"
        )
    inherited = indexed.loc[product_ids]
    if not np.array_equal(
        np.asarray(product["is_low25"].values, dtype=bool),
        inherited["is_low25"].to_numpy(dtype=bool),
    ):
        raise ValueError(
            f"{relative}: low25 membership differs from master ranking"
        )
    if not np.array_equal(
        np.asarray(product["canonical_rank"].values, dtype=int),
        inherited["rank"].to_numpy(dtype=int),
    ):
        raise ValueError(
            f"{relative}: canonical ranks differ from master ranking"
        )
    if not np.allclose(
        np.asarray(product["o3_minimum_du"].values, dtype=float),
        inherited["minimum_du"].to_numpy(dtype=float),
        rtol=0.0, atol=2.0e-5,
    ):
        raise ValueError(
            f"{relative}: O3 minima differ from master ranking"
        )
    ranking_thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if (
        len(ranking_thresholds) != 1
        or not np.isclose(
            float(product.attrs["low25_threshold_du"]),
            float(ranking_thresholds[0]), rtol=0.0, atol=1.0e-10,
        )
    ):
        raise ValueError(
            f"{relative}: threshold differs from master ranking"
        )
    expected_unavailable = sorted(set(scope.index) - set(product_ids))
    recorded_unavailable = sorted(filter(
        None,
        str(product.attrs.get("unavailable_event_ids", "")).split(","),
    ))
    if recorded_unavailable != expected_unavailable:
        raise ValueError(
            f"{relative}: unavailable event IDs differ from ranking join"
        )
    if not np.isfinite(float(product.attrs["low25_threshold_du"])):
        raise ValueError(f"{relative}: non-finite low25 threshold")
    int(product.attrs["bootstrap_seed_z300"])
    int(product.attrs["bootstrap_seed_epflux"])
    method = str(product.attrs.get("epflux_method", "")).lower()
    for token_group in (
        ("ubar",), ("monthly",),
        ("w=none", "w = none", "no omega"),
    ):
        if not any(token in method for token in token_group):
            raise ValueError(
                f"{relative}: epflux_method fails {token_group}"
            )
    expected_master = (46, 11) if expected_label == "merra2" else (230, 57)
    if (
        int(product.attrs["master_sample_size"]) != expected_master[0]
        or int(product.attrs["master_low_count"]) != expected_master[1]
    ):
        raise ValueError(f"{relative}: unexpected canonical master counts")
    available_n = int(product.sizes.get("event_year", -1))
    available_low = int(np.asarray(product["is_low25"].values).sum())
    if (
        int(product.attrs["available_event_count"]) != available_n
        or int(product.attrs["available_low_count"]) != available_low
    ):
        raise ValueError(f"{relative}: availability metadata differs from fields")
    if int(product.attrs["ep_standardization_ddof"]) != 0:
        raise ValueError(f"{relative}: EP standardization must use ddof=0")
    machine_attrs = {
        "natural_month_n2": "True", "do_ubar": "True",
        "w_argument": "None", "wave": "-1",
    }
    for name, expected in machine_attrs.items():
        if str(product.attrs.get(name, "")) != expected:
            raise ValueError(
                f"{relative}: {name}={product.attrs.get(name)!r}; "
                f"expected {expected!r}"
            )
    if "skipna=False" not in str(product.attrs["z300_completeness"]):
        raise ValueError(f"{relative}: incomplete Z300 daily-data gate")
    if expected_label == "waccm":
        if (
            recorded_scope_segments != ["LONGRUN"]
            or len(recorded_excluded) != 23
            or available_n != 206
            or available_low != 51
        ):
            raise ValueError(
                f"{relative}: WACCM field scope must be LONGRUN 206/51 "
                "with 23 BWCN master events excluded"
            )
    if "unavailable_event_ids" not in product.attrs:
        raise ValueError(f"{relative}: unavailable event IDs were not recorded")
    return product


products = (
    (
        "MERRA-2",
        validated_figure15(
            "figure15/merra2_bootstrap5000.nc", "MERRA-2"
        ),
    ),
    (
        "WACCM",
        validated_figure15(
            "figure15/waccm_bootstrap5000.nc", "WACCM"
        ),
    ),
)
months = ("Nov", "Dec", "Jan", "Feb", "Mar")
z_levels = np.arange(-80, 81, 10)
stationary_levels = np.r_[
    np.arange(-320, 0, 40), np.arange(40, 321, 40)
]
ep_levels = np.arange(-1.0, 1.01, 0.2)


def set_polar_boundary(axis):
    theta = np.linspace(0, 2 * np.pi, 200)
    circle = mpath.Path(
        np.column_stack([np.sin(theta), np.cos(theta)]) * 0.5 + 0.5
    )
    axis.set_boundary(circle, transform=axis.transAxes)
    axis.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
    axis.add_feature(
        cfeature.COASTLINE.with_scale("110m"),
        linewidth=0.42, edgecolor="0.3",
    )
    gridlines = axis.gridlines(
        crs=ccrs.PlateCarree(), draw_labels=False,
        linewidth=0.32, color="0.55", alpha=0.42, linestyle=":",
    )
    gridlines.ylocator = mticker.FixedLocator([30, 45, 60, 75])


figure = plt.figure(figsize=(15.8, 16.0))
grid = figure.add_gridspec(
    4, 5, height_ratios=[1.0, 1.10, 1.0, 1.10],
    left=0.055, right=0.92, bottom=0.07, top=0.95,
    hspace=0.40, wspace=0.04,
)
letters = (("a", "b"), ("c", "d"))
for source_index, (label, product) in enumerate(products):
    lon = np.asarray(product["lon"].values, dtype=float)
    lat = np.asarray(product["lat"].values, dtype=float)
    map_axes = []
    map_mappable = None
    for month_index, month in enumerate(months):
        axis = figure.add_subplot(
            grid[source_index * 2, month_index],
            projection=ccrs.NorthPolarStereo(),
        )
        map_axes.append(axis)
        set_polar_boundary(axis)
        anomaly, cyclic_lon = add_cyclic_point(
            np.asarray(
                product["z300_low25_anomaly"].isel(
                    month=month_index
                ).values
            ),
            coord=lon,
        )
        stationary, _ = add_cyclic_point(
            np.asarray(
                product["z300_stationary_climatology"].isel(
                    month=month_index
                ).values
            ),
            coord=lon,
        )
        significant, _ = add_cyclic_point(
            np.asarray(
                product["z300_bootstrap_significant"].isel(
                    month=month_index
                ).values,
                dtype=float,
            ),
            coord=lon,
        )
        map_mappable = axis.contourf(
            cyclic_lon, lat, anomaly, levels=z_levels,
            cmap="RdBu_r", extend="both",
            transform=ccrs.PlateCarree(),
        )
        axis.contour(
            cyclic_lon, lat, stationary,
            levels=stationary_levels, colors="black",
            linewidths=0.45, transform=ccrs.PlateCarree(),
        )
        axis.contourf(
            cyclic_lon, lat, significant,
            levels=[0.5, 1.5], colors="none", hatches=["xx"],
            transform=ccrs.PlateCarree(),
        )
        axis.set_title(month, fontsize=10)
    n_events = int(product.sizes["event_year"])
    n_low = int(np.asarray(product["is_low25"].values).sum())
    map_axes[0].text(
        -0.02, 1.10,
        f"({letters[source_index][0]}) {label}: Z300 low25 "
        f"(available N={n_events}, low={n_low}; "
        f"master N={int(product.attrs['master_sample_size'])})",
        transform=map_axes[0].transAxes, fontweight="bold",
    )
    map_colorbar = figure.colorbar(
        map_mappable, ax=map_axes, pad=0.012, fraction=0.02
    )
    map_colorbar.set_label("Z300 anomaly (m)")

    axis = figure.add_subplot(grid[source_index * 2 + 1, :])
    day = np.asarray(product["season_day"].values, dtype=float)
    pressure = np.asarray(product["pressure"].values, dtype=float)
    ep = np.asarray(
        product["epflux_low25_std_anomaly"].values, dtype=float
    )
    ep_significant = np.asarray(
        product["epflux_bootstrap_significant"].values, dtype=float
    )
    ep_mappable = axis.contourf(
        day, pressure, ep, levels=ep_levels,
        cmap="RdBu_r", extend="both",
    )
    axis.contourf(
        day, pressure, ep_significant, levels=[0.5, 1.5],
        colors="none", hatches=["xx"],
    )
    axis.set_yscale("log")
    axis.invert_yaxis()
    axis.set_ylim(300, 1)
    axis.set_xlim(-0.5, 150.5)
    axis.set_xticks([0, 30, 61, 92, 120], months)
    # A single black 100-hPa reference line replaces the former grid.
    axis.axhline(100.0, color="black", linewidth=0.75, zorder=4)
    axis.set_ylabel("Pressure (hPa)")
    axis.set_title(
        f"({letters[source_index][1]}) {label}: "
        "40–80°N upward EP-flux anomaly",
        loc="left", fontweight="bold",
    )
    ep_colorbar = figure.colorbar(
        ep_mappable, ax=axis, pad=0.012, fraction=0.02
    )
    ep_colorbar.set_label("Standardized anomaly")
figure.suptitle(
    "Low-ozone tropospheric pattern and vertical wave forcing",
    fontsize=15, fontweight="bold",
)
figure.text(
    0.055, 0.018,
    "Hatching: actual low25 composite differs from the mean of "
    "5000 random trajectory composites by at least 2 bootstrap SD.",
    fontsize=8.8,
)
save_figure(figure, "figure15f_ubar_N2correction_bootstrap")
